# Lab 14: cold start, the metrics endpoint, and the thinking switch

Time to ready with and without compilation; the metrics endpoint idle and under a burst of forty; the hybrid Qwen3-4B with thinking off and on.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Lab 10 is read-only on this GPU (no FP8 KV cache); the FP8 checkpoint in Lab 08 runs with unpacked weights.")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
def stream_chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, "stream": True, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    t0 = time.time(); first = None; last = None; gaps = []; text = []
    with urllib.request.urlopen(req, timeout=600) as resp:
        for raw in resp:
            line = raw.decode().strip()
            if not line.startswith("data:") or line == "data: [DONE]": continue
            delta = json.loads(line[5:])["choices"][0].get("delta", {}).get("content")
            if not delta: continue
            now = time.time()
            if first is None: first = now - t0
            else: gaps.append(now - last)
            last = now; text.append(delta)
    return first, gaps, "".join(text)

def metrics(names):
    out = {}
    for line in urllib.request.urlopen(URL + "/metrics", timeout=5).read().decode().splitlines():
        if line.startswith("#"): continue
        for n in names:
            if line.startswith(n + "{") or line.startswith(n + " "):
                try: out[n] = float(line.rsplit(" ", 1)[1])
                except ValueError: pass
    return out
print("streaming and metrics helpers ready")

In [ ]:
# Cell 5: cold start three ways.
for label, extra in (("default", []), ("--enforce-eager", ["--enforce-eager"]), ("default, second start", [])):
    proc, ready_s = serve(["--max-model-len", "4096", *extra]); ll = log_lines("/kaggle/working/serve.log", STARTUP)
    print(f"  {label:<22} ready after {ready_s:6.1f} s   {ll.get('weights', '')[:30]}   {ll.get('graphs', 'no graph capture')[:45]}")
    if label != "default, second start": stop(proc)

In [ ]:
# Cell 6: the metrics endpoint, idle and during a burst of forty.
import threading
NAMES = ["vllm:num_requests_running", "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc", "vllm:num_preemptions_total", "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total", "vllm:generation_tokens_total"]
print("  idle:", {k.split(':')[1]: v for k, v in metrics(NAMES).items()})
sysmsg = "You are the support assistant for a small online shop. " * 20
def one(i): chat([{"role": "system", "content": sysmsg}, {"role": "user", "content": f"Question {i}: tell me about your returns policy in detail."}], max_tokens=300, temperature=0)
ts = [threading.Thread(target=one, args=(i,)) for i in range(40)]; [t.start() for t in ts]
print("  second running waiting kv%  preempt  prefix hit%  gen tokens"); g0 = metrics(NAMES).get(NAMES[6], 0)
for s in range(1, 16):
    time.sleep(1); m = metrics(NAMES)
    print(f"  {s:>6} {int(m.get(NAMES[0], 0)):>7} {int(m.get(NAMES[1], 0)):>7} {100 * m.get(NAMES[2], 0):4.0f} {int(m.get(NAMES[3], 0)):>8} {100 * m.get(NAMES[5], 0) / max(1, m.get(NAMES[4], 1)):10.0f}% {int(m.get(NAMES[6], 0) - g0):>10}")
[t.join() for t in ts]; stop(proc)

In [ ]:
# Cell 7: the thinking switch on the hybrid Qwen3-4B.
MODEL = "Qwen/Qwen3-4B"
proc, ready_s = serve(["--max-model-len", "8192"]); print(f"ready after {ready_s} s")
q = "A shop sells pens at 3 for 2 dollars. How much do 12 pens cost? Answer briefly."
for think in (False, True):
    t0 = time.time(); r = chat([{"role": "user", "content": q}], max_tokens=1500, temperature=0, chat_template_kwargs={"enable_thinking": think}); took = time.time() - t0
    print(f"  thinking {str(think):<5}: {r['usage']['completion_tokens']:5d} completion tokens in {took:5.1f} s   answer: {r['choices'][0]['message']['content'].strip().replace(chr(10), ' ')[-90:]}")
stop(proc)